In [1]:
# Import necessary libraries
import pandas as pd
import os

# Define relative paths based on the project structure
# The ".." tells Python to go up one level from the 'notebooks' folder to the main folder
csv_path = os.path.join("..", "data", "Uber_Eats_data.csv")
json_path = os.path.join("..", "data", "orders.json")

# Load the datasets
print("Loading datasets...")
df = pd.read_csv(csv_path)
orders_df = pd.read_json(json_path)
print("Data loaded successfully!\n")

# Display the initial shapes (number of rows and columns)
print(f"Restaurant Dataset Shape: {df.shape}")
print(f"Orders Dataset Shape: {orders_df.shape}")

Loading datasets...
Data loaded successfully!

Restaurant Dataset Shape: (23193, 13)
Orders Dataset Shape: (25000, 6)


### 1.1 Quick Look at the Restaurant Data
Checking the first few rows, columns, and basic information to understand the data types we are working with.

In [2]:
# Display column names
print("Restaurant Columns:")
print(df.columns.tolist())
print("-" * 50)

# Display data types and non-null counts
print("Restaurant Data Info:")
df.info()

# Preview the first 3 rows
display(df.head(3))

Restaurant Columns:
['name', 'online_order', 'book_table', 'rate', 'votes', 'phone', 'location', 'rest_type', 'dish_liked', 'cuisines', 'approx_cost(for two people)', 'listed_in(type)', 'listed_in(city)']
--------------------------------------------------
Restaurant Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23193 entries, 0 to 23192
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   name                         23193 non-null  object
 1   online_order                 23193 non-null  object
 2   book_table                   23193 non-null  object
 3   rate                         23193 non-null  object
 4   votes                        23193 non-null  int64 
 5   phone                        23193 non-null  object
 6   location                     23193 non-null  object
 7   rest_type                    23193 non-null  object
 8   dish_liked                   23193 non

,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),listed_in(type),listed_in(city)
0,Jalsa,Yes,Yes,4.1/5,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,Buffet,Banashankari
1,Spice Elephant,Yes,No,4.1/5,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,Buffet,Banashankari
2,San Churro Cafe,Yes,No,3.8/5,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,Buffet,Banashankari


### 1.2 Quick Look at the Orders Data (JSON)
Inspecting the generated JSON dataset that contains order histories.

In [3]:
# Display column names
print("Orders Columns:")
print(orders_df.columns.tolist())
print("-" * 50)

# Display data types and non-null counts
print("Orders Data Info:")
orders_df.info()

# Preview the first 3 rows
display(orders_df.head(3))

Orders Columns:
['order_id', 'restaurant_name', 'order_date', 'order_value', 'discount_used', 'payment_method']
--------------------------------------------------
Orders Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         25000 non-null  object 
 1   restaurant_name  25000 non-null  object 
 2   order_date       25000 non-null  object 
 3   order_value      25000 non-null  float64
 4   discount_used    25000 non-null  object 
 5   payment_method   25000 non-null  object 
dtypes: float64(1), object(5)
memory usage: 1.1+ MB


,order_id,restaurant_name,order_date,order_value,discount_used,payment_method
0,8174c2af-07a4-4837-9068-1169d963e36e,TBC Sky Lounge,2025-05-06,201.87,No,Card
1,0f9ddb57-4632-4a9f-9afe-1d2e41b94e32,KC Das - Sweet Paradise,2026-01-08,1392.27,No,Cash
2,099deda6-8c53-41cb-abf8-00126e6b2643,Hotel Kadamba Veg,2026-01-22,1358.35,Yes,UPI


## Step 2: Exploratory Data Analysis (EDA)
Before cleaning, we need to understand the shape of our data, check for duplicates, and identify which columns need transformation from text (object) to numeric formats.

In [4]:
# 1. Check for exact duplicate rows
duplicate_count = df.duplicated().sum()
print(f"Total duplicate rows found: {duplicate_count}")

# 2. Check unique values in categorical columns to understand the market
print("\nTop 5 Locations by Restaurant Count:")
print(df['location'].value_counts().head(5))

print("\nTop 5 Restaurant Types:")
print(df['rest_type'].value_counts().head(5))

Total duplicate rows found: 35

Top 5 Locations by Restaurant Count:
location
Koramangala 5th Block    1783
BTM                      1456
Indiranagar              1350
HSR                      1162
Jayanagar                1037
Name: count, dtype: int64

Top 5 Restaurant Types:
rest_type
Casual Dining         7349
Quick Bites           5240
Cafe                  2325
Dessert Parlor        1076
Casual Dining, Bar     984
Name: count, dtype: int64


## Step 1.5: Exploratory Data Analysis (Pre-Cleaning Inspection)
**Objective:** Analyze the raw data formats, uncover hidden missing values, and understand the distribution of categorical and numerical features before applying cleaning transformations.

In [5]:
print("--- Restaurant Data: Anomaly Detection ---")

# 1. Inspecting the 'rate' column
print("Unique values in 'rate' (First 15):")
print(df['rate'].unique()[:15])
print("\nNotice how ratings have '/5' attached, and contain string anomalies like 'NEW' and '-'.\n")

# 2. Inspecting the 'approx_cost(for two people)' column
print("Unique values in 'approx_cost(for two people)' (First 15):")
print(df['approx_cost(for two people)'].unique()[:15])
print("\nNotice the commas (e.g., '1,200'). This is why pandas read it as a string instead of an integer.\n")

# 3. Categorical Distributions
print("Online Order Distribution:")
print(df['online_order'].value_counts())
print("\nTable Booking Distribution:")
print(df['book_table'].value_counts())

# 4. Numerical Summary (Only 'votes' is numeric right now)
print("\nSummary Statistics for 'votes':")
display(df['votes'].describe())

--- Restaurant Data: Anomaly Detection ---
Unique values in 'rate' (First 15):
['4.1/5' '3.8/5' '3.7/5' '4.6/5' '4.0/5' '4.2/5' '3.9/5' '3.0/5' '3.6/5'
 '2.8/5' '4.4/5' '3.1/5' '4.3/5' '2.6/5' '3.3/5']

Notice how ratings have '/5' attached, and contain string anomalies like 'NEW' and '-'.

Unique values in 'approx_cost(for two people)' (First 15):
['800' '300' '600' '700' '550' '500' '450' '650' '400' '750' '200' '850'
 '1,200' '150' '350']

Notice the commas (e.g., '1,200'). This is why pandas read it as a string instead of an integer.

Online Order Distribution:
online_order
Yes    16358
No      6835
Name: count, dtype: int64

Table Booking Distribution:
book_table
No     17071
Yes     6122
Name: count, dtype: int64

Summary Statistics for 'votes':


count    23193.000000
mean       601.074462
std       1114.854301
min          0.000000
25%        101.000000
50%        221.000000
75%        586.000000
max      16832.000000
Name: votes, dtype: float64

### Inspecting the Orders Dataset

In [6]:
print("--- Orders Data: Baseline Analysis ---")

# 1. Categorical Distributions
print("Payment Methods used:")
print(orders_df['payment_method'].value_counts())
print("\nDiscount Usage:")
print(orders_df['discount_used'].value_counts())

# 2. Numerical Summary of Order Values
print("\nSummary Statistics for 'order_value':")
display(orders_df['order_value'].describe())

# 3. Inspecting Dates
print("\nDate Range (Note: currently stored as strings):")
print(f"Earliest Date String: {orders_df['order_date'].min()}")
print(f"Latest Date String: {orders_df['order_date'].max()}")

--- Orders Data: Baseline Analysis ---
Payment Methods used:
payment_method
Cash    8384
Card    8364
UPI     8252
Name: count, dtype: int64

Discount Usage:
discount_used
No     12509
Yes    12491
Name: count, dtype: int64

Summary Statistics for 'order_value':


count    25000.000000
mean       986.081873
std        472.948544
min        150.050000
25%        589.602500
50%        966.475000
75%       1344.772500
max       1999.840000
Name: order_value, dtype: float64


Date Range (Note: currently stored as strings):
Earliest Date String: 2025-04-29
Latest Date String: 2026-02-23


## Step 2: Data Cleaning & Feature Engineering
**Objective:** 1. Normalize text columns for SQL compatibility.
2. Clean and convert numerical columns (`rate`, `approx_cost_for_two`, `order_date`).
3. Remove duplicates and handle missing values.
4. Perform Feature Engineering to create pricing segments and rating categories.

In [7]:
print("--- Starting Data Cleaning: Restaurant Dataset ---")

# 1. Rename columns to be SQL-friendly
df = df.rename(columns={
    'approx_cost(for two people)': 'approx_cost_for_two',
    'listed_in(type)': 'listed_in_type',
    'listed_in(city)': 'listed_in_city'
})

# 2. Normalize the 'rate' column (Convert "4.1/5" to 4.1)
def clean_rate(x):
    x = str(x).strip()
    if x in ['NEW', '-', 'nan'] or pd.isna(x):
        return None
    try:
        return float(x.split('/')[0])
    except:
        return None

df['rate'] = df['rate'].apply(clean_rate)

# 3. Clean 'approx_cost_for_two' (Remove commas, convert to numeric)
df['approx_cost_for_two'] = df['approx_cost_for_two'].astype(str).str.replace(',', '')
df['approx_cost_for_two'] = pd.to_numeric(df['approx_cost_for_two'], errors='coerce')

# 4. Clean formatting in 'phone' column (Replace newline characters)
df['phone'] = df['phone'].astype(str).str.replace(r'\r\n', ', ', regex=True)

# 5. Handle Duplicates and Missing Values
initial_rows = len(df)
df.drop_duplicates(inplace=True)
df.dropna(inplace=True) # Drops rows where rate or cost became None
final_rows = len(df)

print(f"-> Handled duplicates and nulls. Rows dropped: {initial_rows - final_rows}")

--- Starting Data Cleaning: Restaurant Dataset ---
-> Handled duplicates and nulls. Rows dropped: 181


### 2.1 Feature Engineering
Creating categorical segments out of our continuous numerical data. This will make our SQL `GROUP BY` queries much more powerful for business insights.

In [8]:
print("--- Starting Feature Engineering ---")

# Create Pricing Segments
# Low: < 500, Mid: 500-1000, Premium: > 1000
bins_cost = [0, 500, 1000, 15000]
labels_cost = ['Low', 'Mid', 'Premium']
df['price_segment'] = pd.cut(df['approx_cost_for_two'], bins=bins_cost, labels=labels_cost)

# Create Rating Categories
# Low: < 3.5, Average: 3.5-4.0, Excellent: > 4.0
bins_rate = [0.0, 3.5, 4.0, 5.0]
labels_rate = ['Low', 'Average', 'Excellent']
df['rating_category'] = pd.cut(df['rate'], bins=bins_rate, labels=labels_rate)

print("-> Feature Engineering Complete. New columns added: 'price_segment', 'rating_category'")
display(df[['name', 'rate', 'rating_category', 'approx_cost_for_two', 'price_segment']].head())

--- Starting Feature Engineering ---
-> Feature Engineering Complete. New columns added: 'price_segment', 'rating_category'


,name,rate,rating_category,approx_cost_for_two,price_segment
0,Jalsa,4.1,Excellent,800,Mid
1,Spice Elephant,4.1,Excellent,800,Mid
2,San Churro Cafe,3.8,Average,800,Mid
3,Addhuri Udupi Bhojana,3.7,Average,300,Low
4,Grand Village,3.8,Average,600,Mid


### 2.2 Cleaning the Orders Dataset

In [9]:
print("--- Starting Data Cleaning: Orders Dataset ---")

# 1. Convert order_date to proper DateTime object
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])

# 2. Drop duplicates
initial_orders = len(orders_df)
orders_df.drop_duplicates(inplace=True)

print(f"-> 'order_date' converted to DateTime.")
print(f"-> Duplicates dropped: {initial_orders - len(orders_df)}")
print(f"-> Final Orders Dataset Shape: {orders_df.shape}")

# Final sanity check on datatypes
print("\n--- Final Data Types Check ---")
print("Restaurant Data Types (Key Columns):")
print(df[['rate', 'approx_cost_for_two', 'price_segment']].dtypes)
print("\nOrders Data Types (Key Columns):")
print(orders_df[['order_date', 'order_value']].dtypes)

--- Starting Data Cleaning: Orders Dataset ---
-> 'order_date' converted to DateTime.
-> Duplicates dropped: 0
-> Final Orders Dataset Shape: (25000, 6)

--- Final Data Types Check ---
Restaurant Data Types (Key Columns):
rate                    float64
approx_cost_for_two       int64
price_segment          category
dtype: object

Orders Data Types (Key Columns):
order_date     datetime64[ns]
order_value           float64
dtype: object


## Step 4: Relational Database Migration (SQL)
**Objective:** Store the cleaned DataFrames into a structured SQLite database (`ubereats_analytics.db`). This database will act as the backend engine for the Streamlit dashboard.

In [10]:
import sqlite3
import os

print("--- Starting Database Migration ---")

# 1. Define the database path (Saving it in the 'database' folder)
db_folder = os.path.join("..", "database")
os.makedirs(db_folder, exist_ok=True) # Creates the folder if it doesn't exist
db_path = os.path.join(db_folder, "ubereats_analytics.db")

# 2. Convert categorical and datetime columns to strings for safe SQLite storage
# SQLite does not have strict category or datetime datatypes, so text is best
df['price_segment'] = df['price_segment'].astype(str)
df['rating_category'] = df['rating_category'].astype(str)
orders_df['order_date'] = orders_df['order_date'].astype(str)

# 3. Connect to the SQLite Database
conn = sqlite3.connect(db_path)
print(f"Connected to database at: {db_path}")

# 4. Insert DataFrames into SQL tables
df.to_sql('restaurants', conn, if_exists='replace', index=False)
print(f"-> Inserted {len(df)} rows into the 'restaurants' table.")

orders_df.to_sql('orders', conn, if_exists='replace', index=False)
print(f"-> Inserted {len(orders_df)} rows into the 'orders' table.")

# 5. Verify tables were created
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print(f"\nTables currently in database: {[table[0] for table in tables]}")

--- Starting Database Migration ---
Connected to database at: ..\database\ubereats_analytics.db
-> Inserted 23012 rows into the 'restaurants' table.
-> Inserted 25000 rows into the 'orders' table.

Tables currently in database: ['restaurants', 'orders']


## Step 5: Testing the Database Layer (SQL Query Execution)
**Objective:** Run a test query using `GROUP BY` to ensure the data was inserted correctly and can be aggregated using standard SQL commands.

In [11]:
print("--- Running Test SQL Query ---")

# Let's test our Feature Engineering with a real business question:
# "How do low, mid, and premium-priced restaurants perform in terms of ratings?"

test_query = """
    SELECT 
        price_segment, 
        COUNT(*) as total_restaurants, 
        ROUND(AVG(rate), 2) as average_rating
    FROM restaurants
    WHERE price_segment != 'nan'
    GROUP BY price_segment
    ORDER BY average_rating DESC;
"""

# Execute the query using pandas read_sql to get a clean DataFrame output
test_result = pd.read_sql(test_query, conn)
display(test_result)

# Close the connection
conn.close()
print("Database connection closed. Migration complete!")

--- Running Test SQL Query ---


,price_segment,total_restaurants,average_rating
0,Premium,4677,4.18
1,Mid,8585,3.89
2,Low,9750,3.79


Database connection closed. Migration complete!


In [12]:
# Check how many unique restaurants in the JSON are actually in our cleaned CSV
json_names = set(orders_df['restaurant_name'].unique())
csv_names = set(df['name'].unique())
common = json_names.intersection(csv_names)
print(f"Unique restaurants in JSON: {len(json_names)}")
print(f"Matching restaurants in CSV: {len(common)}")

Unique restaurants in JSON: 3159
Matching restaurants in CSV: 3142


In [13]:
# 1. Ensure order_date is datetime (just in case)
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])

# 2. Check the Start and End of your business data
print("--- Business Time Horizon ---")
print(f"Earliest Order: {orders_df['order_date'].min()}")
print(f"Latest Order:   {orders_df['order_date'].max()}")

# 3. Check for seasonality (Orders per Month)
print("\n--- Monthly Transaction Volume ---")
monthly_counts = orders_df['order_date'].dt.strftime('%B %Y').value_counts()
print(monthly_counts)

# 4. Check for Day-of-Week patterns
print("\n--- Weekly Distribution (0=Monday, 6=Sunday) ---")
print(orders_df['order_date'].dt.dayofweek.value_counts().sort_index())

--- Business Time Horizon ---
Earliest Order: 2025-04-29 00:00:00
Latest Order:   2026-02-23 00:00:00

--- Monthly Transaction Volume ---
order_date
December 2025     2664
January 2026      2642
July 2025         2613
May 2025          2592
October 2025      2531
August 2025       2520
September 2025    2483
November 2025     2478
June 2025         2422
February 2026     1903
April 2025         152
Name: count, dtype: int64

--- Weekly Distribution (0=Monday, 6=Sunday) ---
order_date
0    3613
1    3596
2    3598
3    3623
4    3567
5    3505
6    3498
Name: count, dtype: int64
